# Filter Denver's Mutations

This notebook filters an ANNOVAR-style mutation table by VAF, depth, and
Poisson/duplicate-consistency thresholds.

> **Disclaimer:** This notebook filters Denver's mutations down to what *appears to
> be real mutations*, based on VAF, depth, support by two replicates, and Poisson
> significance. **It does not apply any further filtering for haematopoietic-relevant
> genes or for exonic/functional consequence** (e.g. exonic vs. intronic, synonymous
> vs. non-synonymous). If that kind of filtering is required, apply it yourself on
> the output file.

It is configured to filter Denver's annotated poisson file (**`Poisson_annotated_merged_POOLED_RESTART_180826.tsv`** is the latest with ~6K samples). To run it: place that
file next to this notebook, then **Run All**. One CSV file is written to the `out/`
folder.

Everything is parameterised, so you can point it at a different file by editing one cell.

## 1. Imports

In [1]:
import os
import re
import numpy as np
import pandas as pd

## 2. Parameters
Mirrors the `argparse` block of `FilterMutations.R`. Edit these to change input
file, thresholds, or column names.

In [2]:
run_name="Poisson_annotated_merged_POOLED_RESTART_180826"

P = dict(
    # --- I/O ---
    run_name=run_name,
    input_file=f"data/{run_name}.tsv",               # .xlsx/.xls or tab-delimited .tsv/.txt
    output_dir="out",
    

    # --- column names ---
    gene_column="Gene.refGene",
    chr_column="CHR",
    pos_column="POS",
    sample_column="SAMPLE_ID",
    gnomad_column="gnomad40_genome_AF",          # R uses 'gnomad41_genome_AF'; this file uses 'AF'
    depth_column="DEPTH",        # R dups-mode uses 'DEPTH'
    depth_dup_column="DEPTH_DUP",

    # --- filters ---
    min_VAF=0.005,
    min_depth=20,
    max_gnomad_AF=0.001,
    dups_input_file=True,        # carries VAF_DUP / DEPTH_DUP
    min_avg_VAF=-1,              # -1 => adaptive threshold: (min_depth/10)/(min_depth + mean_depth)
)

os.makedirs(P["output_dir"], exist_ok=True)
def log(msg): print("[LOG]", msg)

## 3. Read input & validate required columns
Mirrors the `fread` + required-column check in `FilterMutations.R`.

In [3]:
if P["input_file"].lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(P["input_file"])
else:
    df = pd.read_csv(P["input_file"], sep="\t")

required = [P["gene_column"], P["sample_column"], P["chr_column"], P["pos_column"],
            "REF", "ALT", "VAF", P["gnomad_column"], "Func.refGene", "AAChange.refGene"]
if P["dups_input_file"]:
    required += ["VAF_DUP", P["depth_dup_column"]]
required += [P["depth_column"]]

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

log(f"Input file {P['input_file']} read: {df.shape[0]} rows, {df.shape[1]} columns")
assert len(df) > 0, "No rows in input!"
df_f = df.copy()

/tmp/ipykernel_1313813/2135256420.py:4: DtypeWarning: Columns (0,34,35,36,39) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(P["input_file"], sep="\t")


[LOG] Input file data/Poisson_annotated_merged_POOLED_RESTART_180826.tsv read: 458750 rows, 57 columns


## 4. Filter by VAF
Handles `%`-string VAFs as the R code does.

In [4]:
vaf_cols = ["VAF", "VAF_DUP"] if P["dups_input_file"] else ["VAF"]
for c in vaf_cols:
    if not pd.api.types.is_numeric_dtype(df_f[c]):
        if df_f[c].astype(str).str.endswith("%").all():
            df_f[c] = df_f[c].astype(str).str.replace("%", "", regex=False).astype(float) / 100
            log(f"Converted {c} percentage strings to numeric.")
        else:
            raise ValueError(f"{c} is not numeric and not percentage strings.")
    df_f = df_f[df_f[c] >= P["min_VAF"]]
    log(f"After {c} >= {P['min_VAF']}: {len(df_f)} rows")
    assert len(df_f) > 0, "No mutations left after VAF filtering!"

[LOG] After VAF >= 0.005: 225708 rows
[LOG] After VAF_DUP >= 0.005: 220552 rows


## 5. Filter by depth

In [5]:
depth_cols = [P["depth_column"], P["depth_dup_column"]] if P["dups_input_file"] else [P["depth_column"]]
for c in depth_cols:
    df_f = df_f[df_f[c] >= P["min_depth"]]
    log(f"After {c} >= {P['min_depth']}: {len(df_f)} rows")
    assert len(df_f) > 0, "No mutations left after depth filtering!"

[LOG] After DEPTH >= 20: 217518 rows
[LOG] After DEPTH_DUP >= 20: 213727 rows


## 6. Extra SNV filters
Stringent, duplicate-aware SNV filtering: keep only mutations where **both** `VAF`
and `VAF_DUP` > 0.006, **both** Poisson BH q-values (`BH1`, `BH2`) < 0.001, and
`dup_ratio` > 10. Non-numeric / missing `BH`/`dup_ratio` values are treated as
failing the filter.

In [6]:
# Thresholds (edit here)
MIN_VAF_BOTH  = 0.006   # require VAF AND VAF_DUP strictly greater than this
MAX_BH        = 0.001   # require BH1 AND BH2 strictly less than this
MIN_DUP_RATIO = 10      # require dup_ratio strictly greater than this

f_vaf_both = (df_f["VAF"] > MIN_VAF_BOTH) & (df_f["VAF_DUP"] > MIN_VAF_BOTH)
f_bh       = (pd.to_numeric(df_f["BH1"], errors="coerce") < MAX_BH) & \
             (pd.to_numeric(df_f["BH2"], errors="coerce") < MAX_BH)
f_dup_ratio = pd.to_numeric(df_f["dup_ratio"], errors="coerce") > MIN_DUP_RATIO

before = len(df_f)
df_f = df_f[f_vaf_both & f_bh & f_dup_ratio]
log(f"After SNV filters (VAF & VAF_DUP > {MIN_VAF_BOTH}, "
    f"BH1 & BH2 < {MAX_BH}, dup_ratio > {MIN_DUP_RATIO}): {len(df_f)} rows (was {before})")
assert len(df_f) > 0, "No mutations left after SNV filters!"

[LOG] After SNV filters (VAF & VAF_DUP > 0.006, BH1 & BH2 < 0.001, dup_ratio > 10): 117202 rows (was 213727)


## 7. Average VAF, adaptive VAF threshold & gnomAD AF filter
Computes `avg_VAF` / `mean_depth`, applies the adaptive (or fixed) minimum average
VAF threshold, then filters by gnomAD population AF.

In [7]:
if P["dups_input_file"]:
    df_f["avg_VAF"] = df_f[["VAF", "VAF_DUP"]].mean(axis=1)
    df_f["mean_depth"] = df_f[[P["depth_column"], P["depth_dup_column"]]].mean(axis=1)
else:
    df_f["avg_VAF"] = df_f["VAF"]
    df_f["mean_depth"] = df_f[P["depth_column"]]

if P["min_avg_VAF"] == -1:
    thr = (P["min_depth"] / 10) / (P["min_depth"] + df_f["mean_depth"])
    f_min_avg_VAF = df_f["avg_VAF"] >= thr
else:
    f_min_avg_VAF = df_f["avg_VAF"] >= P["min_avg_VAF"]

df_f = df_f[f_min_avg_VAF]
log(f"After avg_VAF threshold: {len(df_f)} rows")

df_f["numeric_gnomad_AF"] = pd.to_numeric(df_f[P["gnomad_column"]], errors="coerce")
df_f = df_f[df_f["numeric_gnomad_AF"].isna() | (df_f["numeric_gnomad_AF"] <= P["max_gnomad_AF"])]
log(f"After gnomAD AF <= {P['max_gnomad_AF']}: {len(df_f)} rows")
assert len(df_f) > 0, "No mutations left after avg_VAF/gnomAD filtering!"

[LOG] After avg_VAF threshold: 117202 rows
[LOG] After gnomAD AF <= 0.001: 30664 rows


## 8. Write output file

In [8]:
output_fn = os.path.join(P["output_dir"], f"{P['run_name']}_filtered_SNVs.tsv")

# Count how many rows (samples) carry each variant (CHR:POS:REF:ALT), after all filtering
variant_keys = [P["chr_column"], P["pos_column"], "REF", "ALT"]
df_f["variant_count"] = df_f.groupby(variant_keys)[P["chr_column"]].transform("size")

# Recurrence ratio = variant_count / dup_n_cutoff
df_f["count_dup_ratio"] = df_f["variant_count"] / df_f["dup_n_cutoff"]

# Drop only multi-sample variants (count > 1) whose ratio < 0.3; keep all singletons
drop = (df_f["variant_count"] > 1) & (df_f["count_dup_ratio"] < 0.3)
log(f"Dropped {int(drop.sum())} rows (count>1 & count_dup_ratio<0.3); kept {int((~drop).sum())}")
df_f = df_f[~drop]

df_f.to_csv(output_fn, index=False, na_rep="NA", sep='\t')
log(f"Wrote {output_fn}  ({len(df_f)} rows)")

[LOG] Dropped 9328 rows (count>1 & count_dup_ratio<0.3); kept 21336
[LOG] Wrote out/Poisson_annotated_merged_POOLED_RESTART_180826_filtered_SNVs.tsv  (21336 rows)


## 9. Preview

In [9]:
cols = ["Gene.refGene", "Func.refGene", "ExonicFunc.refGene", "avg_VAF", "variant_count"]
print("Genes:", df_f["Gene.refGene"].value_counts().to_dict())
df_f[cols].head(15)

Genes: {'NF1P2;MIR5701-2': 5420, 'NF1P2': 3751, 'POTEG;OR11H2': 2371, 'LINC02297;POTEM': 1998, 'NF1': 1689, 'OR11H1;POTEH': 872, 'DNMT3A': 870, 'TET2': 502, 'BCOR': 318, 'PDS5B': 247, 'ASXL1': 221, 'BCORL1': 211, 'SMC3': 203, 'TP53': 193, 'EZH2': 188, 'WT1': 176, 'STAG2': 146, 'KDM6A': 144, 'RAD21': 111, 'ZRSR2': 108, 'U2AF1;U2AF1L5': 100, 'PPM1D': 98, 'PLEKHB2;POTEE': 91, 'RUNX1': 91, 'BRCC3': 82, 'ETV6': 79, 'SRSF2': 79, 'CEBPA': 78, 'GATA2': 57, 'CBL': 48, 'JAK2': 47, 'CSF3R': 46, 'CWH43;NONE': 44, 'FLT3': 36, 'GNB1': 36, 'SF3B1': 34, 'NONE;EMB': 33, 'DDX41': 30, 'TET2-AS1': 30, 'KRAS': 29, 'ZNF687': 28, 'MPL': 28, 'CALR': 28, 'SRP19': 27, 'PHF6': 25, 'FRG1BP;DEFB115': 25, 'PTPN11': 25, 'NONE;ALG10B': 21, 'NONE;LOC441666': 18, 'NONE;SPRY3': 16, 'CSF1R': 14, 'IDH1': 14, 'NRAS': 13, 'KIT': 12, 'IDH2': 9, 'ZNF767P;KRBA1': 9, 'BRAF': 9, 'SETBP1': 9, 'SMC1A': 7, 'NPM1': 7, 'ZNF519;ANKRD20A5P': 5, 'TTTY23;GYG2P1': 5, 'GNAS': 5, 'KCNJ18;UBBP4': 5, 'MFSD11': 4, 'SHPK': 4, 'ANKRD20A11P;LIPI'

,Gene.refGene,Func.refGene,ExonicFunc.refGene,avg_VAF,variant_count
31,NRAS,intronic,0,0.540426,1
87,NRAS,exonic,synonymous SNV,0.113695,1
97,NRAS,exonic,nonsynonymous SNV,0.020693,1
148,NRAS,intronic,0,0.518251,1
173,NRAS,intronic,0,0.509323,1
189,NRAS,intronic,0,0.527257,1
226,NRAS,intronic,0,0.026800,1
261,NRAS,exonic,synonymous SNV,0.584855,1
306,NRAS,exonic,synonymous SNV,0.245271,1
332,NRAS,exonic,synonymous SNV,0.321785,1
